Build a compact CNN that ingests the 1-channel FFT log-magnitude image and outputs (1) a binary logit (real vs AI) and (2) a 128-D embedding for future fusion.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np
from PIL import Image
import os

In [3]:
# Steps 1 - 4
class FFTDataset(Dataset):
    def __init__(self, img_paths: list[str], labels: list[int]):
        """
        img_paths (list[str]): image paths
        labels (list[int]): labels for 0 = real, 1 = fake
        """
        self.img_paths = img_paths
        self.labels = labels


    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # get 3 channel image
        img = Image.open(self.img_paths[idx]).convert('RGB')

        # RBG -> grayscale
        img_gray = transforms.functional.rgb_to_grayscale(img)

        # spatial -> frequency
        fft = torch.fft.fft2(img_gray)

        # move 0-frequency component to center of image
        fft_shift = torch.fft.fftshift(fft)

        # log (|1 + FFT|)
        fft_log_mag = torch.log1p(torch.abs(fft_shift))

        # fft log-mag image as float 32 tensory, label
        return fft_log_mag.float(), torch.tensor(self.labels[idx], dtype=torch.float32)

In [4]:
# CNN Model Definition
class CompactFFTNet(nn.Module):
    def __init__(self, input_channels=1, depth=3, base_filters=16, dropout=0.2, embedding_dim=128):
        """ 
        input_channels = 1 because grayscale
        depth : num convolutional blcoks
        base_filters: num filters in first convolution layer
        dropout: dropout rate to prevent overfitting
        embedding_dim: feature embedding vector 
        """
        super().__init__()

        
        layers = []
        in_ch = input_channels      # this will double on each block
        for i in range(depth):
            out_ch = base_filters * (2**i)

            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))   # 2D convolution later
            layers.append(nn.BatchNorm2d(out_ch))                               # normalize feature map?
            layers.append(nn.ReLU())                    
            layers.append(nn.MaxPool2d(2))                                      # reduce dimensions by half
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch

        
        self.cnn = nn.Sequential(*layers)                                       # combine into one module
        self.global_pool = nn.AdaptiveAvgPool2d(1)                              # global average pooling
        self.embedding = nn.Linear(out_ch, embedding_dim)                       # maps layer -> size 128
        self.classifier = nn.Linear(embedding_dim, 1)                           # maps embedding (128) -> single logic bit

    def forward(self, x):
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)      # x: (batch_size, out_ch)
        emb = self.embedding(x)                 # dense layer
        logit = self.classifier(emb)            # x: (batch_size, 1)
        return logit.squeeze(1), emb

In [5]:
def one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # init gradient buffers to 0
        optimizer.zero_grad()

        # bin classification, 128-D embedding
        logits, _ = model(x)

        # find difference between logits and true y label
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return np.mean(losses)

In [ ]:
@torch.inference_mode() # faster for making only predictions
def evaluate(model, loader, device):
  model.eval()

  all_labels = []
  all_preds = []
  all_probs = []
  
  for x, y in loader:
    x, y = x.to(device), y.to(device)

    logits, _ = model(x)
    probs = torch.sigmoid(logits) # probability of fake
    preds = (probs > 0.5).long()  # threshold at 0.5

    all_labels.append(y.cpu())
    all_labels.append(preds.cpu())
    all_probs.append(probs.cpu())
    
  all_labels = torch.cat(all_labels).numpy()
  all_preds = torch.cat(all_preds).numpy()
  all_probs = torch.cat(all_probs).numpy().flatten()

  # getting metrics
  accuracy = accuracy_score(all_labels, all_preds)
  f1 = f1_score(all_labels, all_preds)
  
  # for AUROC, we use probabilities, not predicted classes
  try:
    auroc = roc_auc_score(all_labels, all_probs)
  except ValueError:
    auroc = float('nan') # only if on class is present
  
  return accuracy, f1, auroc

In [7]:
def get_image_paths_and_labels(folder):
    """
    Return list of image paths and binary labels (0 : real, 1 : fake)
    """
    paths = []
    labels = []
    for label, subfolder in enumerate(["real", "fake"]):
        subdir = os.path.join(folder, subfolder)
        for fname in os.listdir(subdir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
    return paths, labels

In [ ]:
def main():
    # Hyperparameters
    data_dir = "datasets/DRAGON"
    epochs = 10
    batch_size = 16
    lr = 1e-3
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # get data
    train_folder = "datasets/DRAGON/train"
    test_folder = "datasets/DRAGON/test"
    train_paths, train_labels = get_image_paths_and_labels(train_folder)
    test_paths, test_labels = get_image_paths_and_labels(test_folder)
    train_dataset = FFTDataset(train_paths, train_labels)
    test_dataset = FFTDataset(test_paths, test_labels)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)   
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    model = CompactFFTNet().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        loss = one_epoch(model, train_loader, optimizer, criterion, device)
        acc, f1, auroc = evaluate(model, test_loader, device)

        
        print(f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {loss:.4f} | "
            f"Acc: {acc:.4f} | "
            f"F1: {f1:.4f} | "
            f"AUROC: {auroc:.4f}")

    torch.save(model.state_dict(), "compact_fft_model.pth")
    print("Training complete. Model saved to compact_fft_model.pth")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/DRAGON/train/real'